## MMA DATABASE TESTING

Connecting to the database and general utility funcs

In [ ]:
import psycopg2
import pandas as pd
import time
from IPython.display import display, Markdown, HTML
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os

def get_connection():
    """Create a connection to the PostgreSQL database in Docker"""
    try:
        load_dotenv('./.env')

        conn = psycopg2.connect(
            host="localhost",
            port=os.getenv("POSTGRES_PORT"),
            database=os.getenv("POSTGRES_DB"),
            user=os.getenv("POSTGRES_USER"),
            password=os.getenv("POSTGRES_PASSWORD")
        )
        return conn
    except Exception as e:
        display(Markdown(f"**Error connecting to database:** {e}"))
        return None

# Test connection
conn = get_connection()
if conn:
    display(Markdown("** Successfully connected to database**"))
    conn.close()
else:
    display(Markdown("** Failed to connect to database**"))

def run_query(query_name, query_sql):
    """Run a query and time its execution"""
    conn = get_connection()
    if not conn:
        return None, -1
    
    cursor = conn.cursor()
    
    # Start timing
    start_time = time.time()
    
    try:
        cursor.execute(query_sql)
        result = cursor.fetchall()
        
        # Create DataFrame with column names
        columns = [desc[0] for desc in cursor.description]
        df = pd.DataFrame(result, columns=columns)
        
    except Exception as e:
        display(Markdown(f"**Error executing query:** {e}"))
        conn.close()
        return None, -1
    
    # End timing
    end_time = time.time()
    execution_time = end_time - start_time
    
    cursor.close()
    conn.close()
    
    # Display results
    display(Markdown(f"## {query_name}"))
    display(Markdown(f"*Execution time: {execution_time:.6f} seconds*"))
    
    return df, execution_time

query_results = {
    "query_name": [],
    "execution_time": []
}

def track_query(name, df, time):
    """Track query results for later comparison"""
    if df is not None and time > 0:
        query_results["query_name"].append(name)
        query_results["execution_time"].append(time)
        return df.shape[0]  # Return row count
    return 0

** Successfully connected to database**

### 1. BASIC DATABASE QUERIES

#### Count records in each table

In [55]:
sql = """
SELECT 'fighters' as table_name, COUNT(*) as record_count FROM fighters
UNION
SELECT 'fights', COUNT(*) FROM fights
UNION
SELECT 'events', COUNT(*) FROM events
UNION
SELECT 'judges', COUNT(*) FROM judges
UNION
SELECT 'fight_scores', COUNT(*) FROM fight_scores
UNION
SELECT 'round_scores', COUNT(*) FROM round_scores
ORDER BY table_name;
"""

df, time_taken = run_query("Table Record Counts", sql)
rows = track_query("Table Record Counts", df, time_taken)
display(df)

## Table Record Counts

*Execution time: 0.031527 seconds*

,table_name,record_count
0,events,716
1,fighters,2262
2,fights,8040
3,fight_scores,9951
4,judges,3
5,round_scores,26418


#### Find events in a given year

In [41]:

sql = """
SELECT *
FROM events
WHERE date >= '2024-01-01' AND date < '2025-01-01'
ORDER BY date;
"""

df, time_taken = run_query("Events in 2024", sql)
rows = track_query("Events in 2024", df, time_taken)

display(df)

## Events in 2024

*Execution time: 0.003011 seconds*

,event_id,date,name,location_name,location,elevation,event_url
0,010986ee359fb863,2024-01-13,Las Vegas,"Las Vegas, Nevada, USA","(1,2)",611.1660766601562,http://ufcstats.com/event-details/010986ee359f...
1,bd85ca0bb4f26cfc,2024-01-20,Toronto,"Toronto, Ontario, Canada","(1,2)",91.7226791381836,http://ufcstats.com/event-details/bd85ca0bb4f2...
2,cce79e827569f26e,2024-02-03,Las Vegas,"Las Vegas, Nevada, USA","(1,2)",611.1660766601562,http://ufcstats.com/event-details/cce79e827569...
3,eaea0fc7b76525a8,2024-02-10,Las Vegas,"Las Vegas, Nevada, USA","(1,2)",611.1660766601562,http://ufcstats.com/event-details/eaea0fc7b765...
4,dab0e6cb8c932162,2024-02-17,Anaheim,"Anaheim, California, USA","(1,2)",47.60859680175781,http://ufcstats.com/event-details/dab0e6cb8c93...
5,902ab9197b83d0db,2024-02-24,Mexico City,"Mexico City, Distrito Federal, Mexico","(1,2)",2229.72216796875,http://ufcstats.com/event-details/902ab9197b83...
6,e4a9dbade7c7e1a7,2024-03-02,Las Vegas,"Las Vegas, Nevada, USA","(1,2)",611.1660766601562,http://ufcstats.com/event-details/e4a9dbade7c7...
7,a9df5ae20a97b090,2024-03-09,Miami,"Miami, Florida, USA","(1,2)",0.5368872284889221,http://ufcstats.com/event-details/a9df5ae20a97...
8,c398235fcaf8d71d,2024-03-16,Las Vegas,"Las Vegas, Nevada, USA","(1,2)",611.1660766601562,http://ufcstats.com/event-details/c398235fcaf8...
9,79ff6545b0abc685,2024-03-23,Las Vegas,"Las Vegas, Nevada, USA","(1,2)",611.1660766601562,http://ufcstats.com/event-details/79ff6545b0ab...


#### Fight details and fighter details

In [47]:
sql = """
SELECT f.fight_id, f.fighter1_id, f.fighter2_id, e.date as fight_date, f.final_method
FROM fights f
JOIN events e ON f.event_id = e.event_id
ORDER BY e.date DESC
LIMIT 10;
"""

df, time_taken = run_query("Last 10 fights", sql)
rows = track_query("Last 10 fights", df, time_taken)

display(df)

sql = """
SELECT *
FROM fights
WHERE fight_id = '85e79748b75eb30e';
"""

df, time_taken = run_query("Pulling fights and fighter details", sql)
rows = track_query("Pulling fights and fighter details", df, time_taken)

display(df)

sql = """
SELECT *
FROM fighters
WHERE fighter_id = '1338e2c7480bdf9e';
"""

df, time_taken = run_query("Pulling fights and fighter details", sql)
rows = track_query("Pulling fights and fighter details", df, time_taken)

display(df)

## Last 10 fights

*Execution time: 0.014156 seconds*

,fight_id,fighter1_id,fighter2_id,fight_date,final_method
0,d13849f49f99bf01,0d7b51c9d2649a6e,0d8011111be000b2,2025-02-08,Decision - Unanimous
1,85e79748b75eb30e,1338e2c7480bdf9e,881bf86d4cba8578,2025-02-01,KO/TKO
2,daef1691c7d6b1e4,275aca31f61ba28c,b6452706b373eea1,2025-01-18,Submission
3,f46308108eb9261a,7447e9f28508106a,beecb672a279223e,2025-01-11,Submission
4,00c6a2ef07ca51da,dc9572dd6ec74859,b9437600497350f3,2024-12-14,TKO - Doctor's Stoppage
5,e761c5009c09b295,a0f0004aadf10b71,d33da8a3d82bdb62,2024-12-07,Submission
6,f53573316a4f349f,d661ce4da776fc20,aa72b0f831d0bfe5,2024-11-23,Decision - Unanimous
7,b35e47f2f58ef026,07f72a2a7591b409,d28dee5c705991df,2024-11-16,KO/TKO
8,73f451d894a1d4ca,84b3e7d38f2d2ec5,7ee0fd831c0fe7c3,2024-11-09,KO/TKO
9,799acf3ae8a5df0a,792be9a24df82ed6,6d35bf94f7d30241,2024-11-02,Decision - Unanimous


## Pulling fights and fighter details

*Execution time: 0.002031 seconds*

,fight_id,event_id,fight_url,status,curr_round,curr_time,last_updated,referee,fighter1_id,fighter2_id,final_method
0,85e79748b75eb30e,80dbeb1dd5b53e64,http://ufcstats.com/fight-details/85e79748b75e...,finished,None,None,None,Marc Goddard,1338e2c7480bdf9e,881bf86d4cba8578,KO/TKO


## Pulling fights and fighter details

*Execution time: 0.001935 seconds*

,fighter_id,name,nickname,dob,height,weight,reach,stance
0,1338e2c7480bdf9e,Israel Adesanya,The Last Stylebender,1989-07-22,76,185,80,switch


#### Stance distribution

In [20]:
# Stance distribution
sql = """
SELECT stance, COUNT(*) as count
FROM fighters
GROUP BY stance
ORDER BY count DESC;
"""

df, time_taken = run_query("Fighter Stances Distribution", sql)
rows = track_query("Fighter Stances Distribution", df, time_taken)
display(df)



## Fighter Stances Distribution

*Execution time: 0.001861 seconds*

,stance,count
0,orthodox,1692
1,southpaw,384
2,switch,113
3,None,64
4,open stance,6
5,sideways,3


### 2. COMPLEX QUERIES

In [57]:
sql = """
SELECT fighter_id,
       fighter_name,
       wins,
       losses,
       CASE WHEN (wins + losses) > 0 THEN ROUND(wins * 100.0 / (wins + losses), 2) ELSE 0 END AS win_percentage
FROM (
  SELECT fighter_id,
         fighter_name,
         SUM(CASE WHEN result = 'W' THEN 1 ELSE 0 END) AS wins,
         SUM(CASE WHEN result = 'L' THEN 1 ELSE 0 END) AS losses
  FROM (
    SELECT f.fighter1_id AS fighter_id,
           fighter.name AS fighter_name,
           f.fighter1_result AS result
    FROM fights f
    JOIN fighters fighter ON f.fighter1_id = fighter.fighter_id
    WHERE f.fighter1_result IS NOT NULL
    UNION ALL
    SELECT f.fighter2_id AS fighter_id,
           fighter.name AS fighter_name,
           f.fighter2_result AS result
    FROM fights f
    JOIN fighters fighter ON f.fighter2_id = fighter.fighter_id
    WHERE f.fighter2_result IS NOT NULL
  ) combined
  GROUP BY fighter_id, fighter_name
) stats
WHERE wins + losses > 2  -- Only fighters with at least 3 fights
ORDER BY win_percentage DESC, wins DESC
LIMIT 20;
"""

df, time_taken = run_query("Win percentages", sql)
rows = track_query("Win percentages", df, time_taken)
display(df)


**Error connecting to database:** connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?


None

In [65]:
sql = """
WITH fighter_results AS (
    SELECT 
        fighter_id,
        result,
        e.date AS fight_date,
        ROW_NUMBER() OVER (PARTITION BY fighter_id ORDER BY e.date DESC) AS fight_number
    FROM (
        -- Fighter 1 results
        SELECT 
            fighter1_id AS fighter_id, 
            fighter1_result AS result,
            event_id
        FROM fights
        WHERE fighter1_result IS NOT NULL
        
        UNION ALL
        
        -- Fighter 2 results
        SELECT 
            fighter2_id AS fighter_id,
            fighter2_result AS result,
            event_id
        FROM fights
        WHERE fighter2_result IS NOT NULL
    ) AS combined_results
    JOIN events e ON combined_results.event_id = e.event_id
),
win_streaks AS (
    SELECT 
        fighter_id,
        SUM(CASE WHEN result = 'W' THEN 1 ELSE 0 END) 
            OVER (PARTITION BY fighter_id ORDER BY fight_date DESC 
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS wins_so_far,
        SUM(CASE WHEN result != 'W' THEN 1 ELSE 0 END) 
            OVER (PARTITION BY fighter_id ORDER BY fight_date DESC 
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS non_wins_so_far,
        fight_number
    FROM fighter_results
),
current_streaks AS (
    SELECT 
        fighter_id,
        MAX(CASE WHEN non_wins_so_far = 0 THEN wins_so_far ELSE 0 END) AS current_win_streak
    FROM win_streaks
    GROUP BY fighter_id
)
SELECT 
    f.name,
    f.stance,
    f.height,
    f.weight,
    f.reach,
    COUNT(DISTINCT CASE WHEN fr.result = 'W' THEN fr.fight_date END) AS wins,
    COUNT(DISTINCT CASE WHEN fr.result = 'L' THEN fr.fight_date END) AS losses,
    COUNT(DISTINCT CASE WHEN fr.result NOT IN ('W', 'L') THEN fr.fight_date END) AS other_results,
    cs.current_win_streak,
    ROUND(AVG(CASE WHEN fs.fighter_id = f.fighter_id THEN fs.sig_strikes_landed END), 2) AS avg_sig_strikes_landed,
    ROUND(AVG(CASE WHEN fs.fighter_id = f.fighter_id THEN fs.td_landed END), 2) AS avg_takedowns_landed
FROM fighters f
LEFT JOIN fighter_results fr ON f.fighter_id = fr.fighter_id
LEFT JOIN current_streaks cs ON f.fighter_id = cs.fighter_id
LEFT JOIN fight_stats fs ON f.fighter_id = fs.fighter_id
GROUP BY f.fighter_id, f.name, f.stance, f.height, f.weight, f.reach, cs.current_win_streak
HAVING COUNT(fr.fighter_id) >= 3
ORDER BY cs.current_win_streak DESC, wins DESC
LIMIT 25;
"""

df, time_taken = run_query("Longest win streaks", sql)
rows = track_query("Longest win streaks", df, time_taken)
display(df)


## Longest win streaks

*Execution time: 0.104887 seconds*

,name,stance,height,weight,reach,wins,losses,other_results,current_win_streak,avg_sig_strikes_landed,avg_takedowns_landed
0,Islam Makhachev,southpaw,70,155,70,16,1,0,15,26.76,2.18
1,Georges St-Pierre,orthodox,71,185,76,20,2,0,13,59.68,4.09
2,Khabib Nurmagomedov,orthodox,70,155,70,13,0,0,13,54.23,4.69
3,Merab Dvalishvili,orthodox,66,135,68,12,2,0,12,73.00,6.57
4,Dricus Du Plessis,switch,73,185,76,9,0,0,9,84.11,2.33
5,Ilia Topuria,orthodox,67,145,69,8,0,0,8,48.00,1.38
6,Movsar Evloev,orthodox,67,145,72,8,0,0,8,57.13,5.00
7,Khamzat Chimaev,orthodox,74,185,75,8,0,0,8,32.63,1.75
8,Alexandre Pantoja,orthodox,65,125,67,13,3,0,7,55.81,2.31
9,Mario Bautista,switch,69,135,69,9,2,0,7,49.27,1.18


In [67]:
sql = """
WITH fight_details AS (
    SELECT 
        f.fight_id,
        e.date AS fight_date,
        f.final_method,
        f1.name AS fighter1_name,
        f2.name AS fighter2_name,
        f1.height AS fighter1_height,
        f2.height AS fighter2_height,
        f1.weight AS fighter1_weight,
        f2.weight AS fighter2_weight,
        f1.reach AS fighter1_reach,
        f2.reach AS fighter2_reach,
        ABS(f1.height - f2.height) AS height_diff,
        ABS(f1.reach - f2.reach) AS reach_diff,
        fs1.sig_strikes_landed AS fighter1_sig_strikes,
        fs2.sig_strikes_landed AS fighter2_sig_strikes,
        fs1.td_landed AS fighter1_takedowns,
        fs2.td_landed AS fighter2_takedowns,
        f.fighter1_result,
        f.fighter2_result,
        CASE 
            WHEN f1.weight BETWEEN 125 AND 135 THEN 'Flyweight/Bantamweight'
            WHEN f1.weight BETWEEN 136 AND 145 THEN 'Featherweight'
            WHEN f1.weight BETWEEN 146 AND 155 THEN 'Lightweight'
            WHEN f1.weight BETWEEN 156 AND 170 THEN 'Welterweight'
            WHEN f1.weight BETWEEN 171 AND 185 THEN 'Middleweight'
            WHEN f1.weight BETWEEN 186 AND 205 THEN 'Light Heavyweight'
            WHEN f1.weight > 205 THEN 'Heavyweight'
            ELSE 'Unknown'
        END AS weight_class
    FROM fights f
    JOIN events e ON f.event_id = e.event_id
    JOIN fighters f1 ON f.fighter1_id = f1.fighter_id
    JOIN fighters f2 ON f.fighter2_id = f2.fighter_id
    LEFT JOIN fight_stats fs1 ON f.fight_id = fs1.fight_id AND f.fighter1_id = fs1.fighter_id
    LEFT JOIN fight_stats fs2 ON f.fight_id = fs2.fight_id AND f.fighter2_id = fs2.fighter_id
    WHERE f.fighter1_result IS NOT NULL
    AND f.fighter2_result IS NOT NULL
)
SELECT 
    weight_class,
    COUNT(*) AS total_fights,
    ROUND(AVG(height_diff), 2) AS avg_height_diff_inches,
    ROUND(AVG(reach_diff), 2) AS avg_reach_diff_inches,
    ROUND(AVG(fighter1_sig_strikes + fighter2_sig_strikes), 2) AS avg_total_sig_strikes,
    ROUND(AVG(fighter1_takedowns + fighter2_takedowns), 2) AS avg_total_takedowns,
    COUNT(CASE WHEN final_method LIKE '%KO%' OR final_method LIKE '%TKO%' THEN 1 END) AS ko_tko_wins,
    COUNT(CASE WHEN final_method LIKE '%Submission%' THEN 1 END) AS submission_wins,
    COUNT(CASE WHEN final_method LIKE '%Decision%' THEN 1 END) AS decision_wins,
    ROUND(COUNT(CASE WHEN final_method LIKE '%KO%' OR final_method LIKE '%TKO%' THEN 1 END)::numeric / COUNT(*) * 100, 2) AS ko_tko_percentage,
    ROUND(COUNT(CASE WHEN final_method LIKE '%Submission%' THEN 1 END)::numeric / COUNT(*) * 100, 2) AS submission_percentage,
    ROUND(COUNT(CASE WHEN final_method LIKE '%Decision%' THEN 1 END)::numeric / COUNT(*) * 100, 2) AS decision_percentage,
    ROUND(COUNT(CASE WHEN (fighter1_height > fighter2_height AND fighter1_result = 'W') OR 
                         (fighter2_height > fighter1_height AND fighter2_result = 'W') THEN 1 END)::numeric / 
          COUNT(CASE WHEN fighter1_height != fighter2_height THEN 1 END) * 100, 2) AS taller_fighter_win_percentage,
    ROUND(COUNT(CASE WHEN (fighter1_reach > fighter2_reach AND fighter1_result = 'W') OR 
                         (fighter2_reach > fighter1_reach AND fighter2_result = 'W') THEN 1 END)::numeric / 
          COUNT(CASE WHEN fighter1_reach != fighter2_reach THEN 1 END) * 100, 2) AS longer_reach_win_percentage
FROM fight_details
GROUP BY weight_class
ORDER BY total_fights DESC;
"""

df, time_taken = run_query("Weight class analysis", sql)
rows = track_query("Weight class analysis", df, time_taken)
display(df)

## Weight class analysis

*Execution time: 0.040069 seconds*

,weight_class,total_fights,avg_height_diff_inches,avg_reach_diff_inches,avg_total_sig_strikes,avg_total_takedowns,ko_tko_wins,submission_wins,decision_wins,ko_tko_percentage,submission_percentage,decision_percentage,taller_fighter_win_percentage,longer_reach_win_percentage
0,Flyweight/Bantamweight,1394,1.83,2.44,87.29,2.37,328,262,786,23.53,18.79,56.38,48.20,49.26
1,Welterweight,1386,1.96,2.60,69.46,2.19,435,281,653,31.39,20.27,47.11,52.32,52.09
2,Lightweight,1162,1.94,2.28,70.32,2.31,344,244,559,29.60,21.00,48.11,49.16,48.84
3,Middleweight,1013,1.84,2.48,65.95,2.03,386,207,402,38.10,20.43,39.68,53.11,52.25
4,Featherweight,752,2.04,2.39,82.07,2.30,219,144,378,29.12,19.15,50.27,48.49,50.76
5,Heavyweight,745,2.53,3.36,54.77,1.32,385,137,212,51.68,18.39,28.46,54.11,58.97
6,Light Heavyweight,686,1.87,2.49,57.67,1.92,295,134,243,43.00,19.53,35.42,52.89,55.44
7,Unknown,273,1.99,2.54,101.22,2.26,31,59,181,11.36,21.61,66.30,50.43,41.10


In [71]:
sql = """
WITH fighter_fight_sequence AS (
    SELECT 
        f.fighter_id,
        f.name AS fighter_name,
        ft.fight_id,
        e.date AS fight_date,
        CASE 
            WHEN ft.fighter1_id = f.fighter_id THEN ft.fighter1_result
            ELSE ft.fighter2_result
        END AS result,
        CASE
            WHEN ft.fighter1_id = f.fighter_id THEN fs1.sig_strikes_landed
            ELSE fs2.sig_strikes_landed
        END AS sig_strikes_landed,
        CASE
            WHEN ft.fighter1_id = f.fighter_id THEN fs1.sig_strikes_attempted
            ELSE fs2.sig_strikes_attempted
        END AS sig_strikes_attempted,
        CASE
            WHEN ft.fighter1_id = f.fighter_id THEN fs1.td_landed
            ELSE fs2.td_landed
        END AS td_landed,
        CASE
            WHEN ft.fighter1_id = f.fighter_id THEN fs1.td_attempted
            ELSE fs2.td_attempted
        END AS td_attempted,
        CASE
            WHEN ft.fighter1_id = f.fighter_id THEN fs1.ctrl
            ELSE fs2.ctrl
        END AS control_time_seconds,
        ROW_NUMBER() OVER (PARTITION BY f.fighter_id ORDER BY e.date) AS fight_number,
        COUNT(*) OVER (PARTITION BY f.fighter_id) AS total_fights
    FROM fighters f
    JOIN fights ft ON f.fighter_id IN (ft.fighter1_id, ft.fighter2_id)
    JOIN events e ON ft.event_id = e.event_id
    LEFT JOIN fight_stats fs1 ON ft.fight_id = fs1.fight_id AND ft.fighter1_id = fs1.fighter_id
    LEFT JOIN fight_stats fs2 ON ft.fight_id = fs2.fight_id AND ft.fighter2_id = fs2.fighter_id
    WHERE (ft.fighter1_result IS NOT NULL OR ft.fighter2_result IS NOT NULL)
),
fighter_performance_trends AS (
    SELECT
        fighter_id,
        fighter_name,
        AVG(CASE WHEN fight_number <= total_fights/3 THEN 
                COALESCE(sig_strikes_landed::numeric / NULLIF(sig_strikes_attempted, 0), 0) 
            END) AS early_career_striking_accuracy,
        AVG(CASE WHEN fight_number > total_fights/3 AND fight_number <= 2*total_fights/3 THEN 
                COALESCE(sig_strikes_landed::numeric / NULLIF(sig_strikes_attempted, 0), 0) 
            END) AS mid_career_striking_accuracy,
        AVG(CASE WHEN fight_number > 2*total_fights/3 THEN 
                COALESCE(sig_strikes_landed::numeric / NULLIF(sig_strikes_attempted, 0), 0) 
            END) AS late_career_striking_accuracy,
        AVG(CASE WHEN fight_number <= total_fights/3 THEN 
                COALESCE(td_landed::numeric / NULLIF(td_attempted, 0), 0) 
            END) AS early_career_td_accuracy,
        AVG(CASE WHEN fight_number > total_fights/3 AND fight_number <= 2*total_fights/3 THEN 
                COALESCE(td_landed::numeric / NULLIF(td_attempted, 0), 0) 
            END) AS mid_career_td_accuracy,
        AVG(CASE WHEN fight_number > 2*total_fights/3 THEN 
                COALESCE(td_landed::numeric / NULLIF(td_attempted, 0), 0) 
            END) AS late_career_td_accuracy,
        AVG(CASE WHEN fight_number <= total_fights/3 THEN control_time_seconds END) AS early_career_control_time,
        AVG(CASE WHEN fight_number > total_fights/3 AND fight_number <= 2*total_fights/3 THEN control_time_seconds END) AS mid_career_control_time,
        AVG(CASE WHEN fight_number > 2*total_fights/3 THEN control_time_seconds END) AS late_career_control_time,
        SUM(CASE WHEN fight_number <= total_fights/3 AND result = 'W' THEN 1 ELSE 0 END)::numeric / 
            NULLIF(SUM(CASE WHEN fight_number <= total_fights/3 THEN 1 ELSE 0 END), 0) AS early_career_win_rate,
        SUM(CASE WHEN fight_number > total_fights/3 AND fight_number <= 2*total_fights/3 AND result = 'W' THEN 1 ELSE 0 END)::numeric / 
            NULLIF(SUM(CASE WHEN fight_number > total_fights/3 AND fight_number <= 2*total_fights/3 THEN 1 ELSE 0 END), 0) AS mid_career_win_rate,
        SUM(CASE WHEN fight_number > 2*total_fights/3 AND result = 'W' THEN 1 ELSE 0 END)::numeric / 
            NULLIF(SUM(CASE WHEN fight_number > 2*total_fights/3 THEN 1 ELSE 0 END), 0) AS late_career_win_rate
    FROM fighter_fight_sequence
    GROUP BY fighter_id, fighter_name, total_fights
    HAVING COUNT(*) >= 6
)
SELECT * FROM (
    SELECT
        fighter_name,
        ROUND(early_career_striking_accuracy * 100, 2) AS early_striking_pct,
        ROUND(mid_career_striking_accuracy * 100, 2) AS mid_striking_pct,
        ROUND(late_career_striking_accuracy * 100, 2) AS late_striking_pct,
        ROUND((late_career_striking_accuracy - early_career_striking_accuracy) * 100, 2) AS striking_accuracy_change,
        ROUND(early_career_td_accuracy * 100, 2) AS early_td_pct,
        ROUND(mid_career_td_accuracy * 100, 2) AS mid_td_pct,
        ROUND(late_career_td_accuracy * 100, 2) AS late_td_pct,
        ROUND((late_career_td_accuracy - early_career_td_accuracy) * 100, 2) AS td_accuracy_change,
        ROUND(early_career_control_time, 2) AS early_control_time_avg,
        ROUND(mid_career_control_time, 2) AS mid_control_time_avg,
        ROUND(late_career_control_time, 2) AS late_control_time_avg,
        ROUND((late_career_control_time - early_career_control_time), 2) AS control_time_change,
        ROUND(early_career_win_rate * 100, 2) AS early_win_pct,
        ROUND(mid_career_win_rate * 100, 2) AS mid_win_pct,
        ROUND(late_career_win_rate * 100, 2) AS late_win_pct,
        ROUND((late_career_win_rate - early_career_win_rate) * 100, 2) AS win_rate_change
    FROM fighter_performance_trends
    WHERE early_career_win_rate IS NOT NULL AND late_career_win_rate IS NOT NULL
) AS fighter_data
ORDER BY ABS(win_rate_change) DESC
LIMIT 25;
"""

df, time_taken = run_query("Fighter performance trajectory", sql)
rows = track_query("Fighter performance trajectory", df, time_taken)
display(df)

## Fighter performance trajectory

*Execution time: 0.078699 seconds*

,fighter_name,early_striking_pct,mid_striking_pct,late_striking_pct,striking_accuracy_change,early_td_pct,mid_td_pct,late_td_pct,td_accuracy_change,early_control_time_avg,mid_control_time_avg,late_control_time_avg,control_time_change,early_win_pct,mid_win_pct,late_win_pct,win_rate_change
0,Jason Lambert,61.62,57.08,34.33,-27.29,25.00,33.33,33.33,8.33,113.50,137.00,101.33,-12.17,100.00,66.67,0.00,-100.00
1,Zach Makovsky,41.81,49.93,44.31,2.50,53.36,39.77,20.35,-33.01,243.00,423.00,71.67,-171.33,100.00,50.00,0.00,-100.00
2,Herbert Burns,45.59,64.39,28.35,-17.24,100.00,62.50,11.11,-88.89,66.50,95.50,95.00,28.50,100.00,0.00,0.00,-100.00
3,Marco Ruas,78.57,62.97,63.21,-15.36,50.00,25.00,50.00,0.00,0.00,0.00,110.50,110.50,100.00,100.00,0.00,-100.00
4,Phil Hawes,60.52,62.04,50.39,-10.13,28.57,27.78,33.33,4.76,333.50,127.00,15.33,-318.17,100.00,66.67,0.00,-100.00
5,Francis Carmont,70.75,33.93,31.93,-38.83,50.00,39.39,8.33,-41.67,240.00,386.33,42.33,-197.67,100.00,100.00,0.00,-100.00
6,Devonte Smith,56.30,53.93,49.70,-6.61,0.00,50.00,0.00,0.00,2.50,94.00,7.00,4.50,100.00,50.00,0.00,-100.00
7,Akira Corassani,52.10,43.18,35.09,-17.01,16.67,0.00,0.00,-16.67,136.00,6.00,6.00,-130.00,100.00,50.00,0.00,-100.00
8,Valerie Letourneau,34.23,39.84,43.33,9.10,50.00,60.00,50.00,0.00,28.00,217.00,70.50,42.50,100.00,50.00,0.00,-100.00
9,Jason Brilz,56.98,37.00,47.90,-9.08,75.00,25.00,7.69,-67.31,462.50,326.00,99.00,-363.50,100.00,50.00,0.00,-100.00


In [70]:
sql = """
WITH fight_outcomes AS (
    SELECT 
        f.fight_id,
        f.fighter1_id,
        f.fighter2_id,
        CASE 
            WHEN f.fighter1_result = 'W' THEN f.fighter1_id
            WHEN f.fighter2_result = 'W' THEN f.fighter2_id
            ELSE NULL
        END AS winner_id,
        f.final_method,
        e.date AS fight_date
    FROM fights f
    JOIN events e ON f.event_id = e.event_id
    WHERE f.fighter1_result IS NOT NULL AND f.fighter2_result IS NOT NULL
),
fighter_attributes AS (
    SELECT 
        f.fighter_id,
        f.height,
        f.reach,
        f.stance,
        EXTRACT(YEAR FROM AGE(fo.fight_date, f.dob)) AS age_at_fight,
        fo.fight_id,
        fo.winner_id = f.fighter_id AS is_winner,
        f.fighter_id = fo.fighter1_id AS is_fighter1,
        fo.final_method
    FROM fighters f
    JOIN fight_outcomes fo ON f.fighter_id IN (fo.fighter1_id, fo.fighter2_id)
),
fight_stats_summary AS (
    SELECT
        fs.fight_id,
        fs.fighter_id,
        fs.sig_strikes_landed,
        fs.sig_strikes_accuracy,
        fs.td_landed,
        fs.td_accuracy,
        fs.ctrl
    FROM fight_stats fs
),
paired_attributes AS (
    SELECT 
        fa1.fight_id,
        fa1.fighter_id AS fighter1_id,
        fa2.fighter_id AS fighter2_id,
        fa1.height AS fighter1_height,
        fa2.height AS fighter2_height,
        fa1.reach AS fighter1_reach,
        fa2.reach AS fighter2_reach,
        fa1.age_at_fight AS fighter1_age,
        fa2.age_at_fight AS fighter2_age,
        fa1.stance AS fighter1_stance,
        fa2.stance AS fighter2_stance,
        fa1.is_winner AS fighter1_is_winner,
        fo.final_method,
        fss1.sig_strikes_landed AS fighter1_sig_strikes,
        fss2.sig_strikes_landed AS fighter2_sig_strikes,
        fss1.td_landed AS fighter1_td_landed,
        fss2.td_landed AS fighter2_td_landed,
        fss1.ctrl AS fighter1_ctrl,
        fss2.ctrl AS fighter2_ctrl
    FROM fighter_attributes fa1
    JOIN fighter_attributes fa2 ON fa1.fight_id = fa2.fight_id 
                              AND fa1.fighter_id != fa2.fighter_id
                              AND fa1.is_fighter1 = TRUE 
                              AND fa2.is_fighter1 = FALSE
    JOIN fight_outcomes fo ON fa1.fight_id = fo.fight_id
    LEFT JOIN fight_stats_summary fss1 ON fa1.fight_id = fss1.fight_id AND fa1.fighter_id = fss1.fighter_id
    LEFT JOIN fight_stats_summary fss2 ON fa2.fight_id = fss2.fight_id AND fa2.fighter_id = fss2.fighter_id
)
SELECT
    CASE 
        WHEN fighter1_height > fighter2_height THEN 'Taller Fighter'
        WHEN fighter1_height < fighter2_height THEN 'Shorter Fighter'
        ELSE 'Equal Height'
    END AS height_advantage,
    COUNT(*) AS total_fights,
    SUM(CASE WHEN fighter1_is_winner AND fighter1_height > fighter2_height THEN 1
             WHEN NOT fighter1_is_winner AND fighter2_height > fighter1_height THEN 1
             ELSE 0 END) AS taller_fighter_wins,
    ROUND(100.0 * SUM(CASE WHEN fighter1_is_winner AND fighter1_height > fighter2_height THEN 1
                           WHEN NOT fighter1_is_winner AND fighter2_height > fighter1_height THEN 1
                           ELSE 0 END) / COUNT(*), 2) AS taller_win_percentage,
    
    ROUND(AVG(CASE WHEN fighter1_is_winner THEN fighter1_sig_strikes ELSE fighter2_sig_strikes END), 2) AS avg_winner_sig_strikes,
    ROUND(AVG(CASE WHEN NOT fighter1_is_winner THEN fighter1_sig_strikes ELSE fighter2_sig_strikes END), 2) AS avg_loser_sig_strikes,
    
    ROUND(AVG(CASE WHEN fighter1_is_winner THEN fighter1_td_landed ELSE fighter2_td_landed END), 2) AS avg_winner_takedowns,
    ROUND(AVG(CASE WHEN NOT fighter1_is_winner THEN fighter1_td_landed ELSE fighter2_td_landed END), 2) AS avg_loser_takedowns,
    
    ROUND(AVG(CASE WHEN fighter1_is_winner THEN fighter1_ctrl ELSE fighter2_ctrl END), 2) AS avg_winner_control_time,
    ROUND(AVG(CASE WHEN NOT fighter1_is_winner THEN fighter1_ctrl ELSE fighter2_ctrl END), 2) AS avg_loser_control_time,
    
    COUNT(CASE WHEN final_method LIKE '%KO%' OR final_method LIKE '%TKO%' THEN 1 END) AS ko_tko_wins,
    COUNT(CASE WHEN final_method LIKE '%ubmission%' THEN 1 END) AS submission_wins,
    COUNT(CASE WHEN final_method LIKE '%ecision%' THEN 1 END) AS decision_wins,
    
    ROUND(100.0 * COUNT(CASE WHEN final_method LIKE '%KO%' OR final_method LIKE '%TKO%' THEN 1 END) / COUNT(*), 2) AS ko_tko_percentage,
    ROUND(100.0 * COUNT(CASE WHEN final_method LIKE '%ubmission%' THEN 1 END) / COUNT(*), 2) AS submission_percentage,
    ROUND(100.0 * COUNT(CASE WHEN final_method LIKE '%ecision%' THEN 1 END) / COUNT(*), 2) AS decision_percentage
FROM paired_attributes
WHERE fighter1_height IS NOT NULL AND fighter2_height IS NOT NULL
GROUP BY height_advantage
HAVING COUNT(*) > 5
ORDER BY total_fights DESC;
"""

df, time_taken = run_query("Feature analysis", sql)
rows = track_query("Feature analysis", df, time_taken)
display(df)

## Feature analysis

*Execution time: 0.930726 seconds*

,height_advantage,total_fights,taller_fighter_wins,taller_win_percentage,avg_winner_sig_strikes,avg_loser_sig_strikes,avg_winner_takedowns,avg_loser_takedowns,avg_winner_control_time,avg_loser_control_time,ko_tko_wins,submission_wins,decision_wins,ko_tko_percentage,submission_percentage,decision_percentage
0,Taller Fighter,3081,2046,66.41,43.55,28.47,1.38,0.68,175.15,85.97,1018,621,1384,33.04,20.16,44.92
1,Shorter Fighter,3067,1086,35.41,43.62,29.34,1.54,0.64,187.29,79.17,1000,598,1431,32.61,19.50,46.66
2,Equal Height,1248,0,0.00,43.17,29.00,1.44,0.70,182.68,82.24,398,242,598,31.89,19.39,47.92
